In [0]:
%run ../00_config/project_config

database,volume_name
raw,github_volume


In [0]:
source_path = (
    f"{RAW_VOLUME_PATH}/github_events"
)

schema_path = (
    f"{RAW_VOLUME_PATH}/schemas/github_events"
)

checkpoint_path = (
    f"{RAW_VOLUME_PATH}/checkpoints/bronze"
)

In [0]:
raw_stream = (
    spark.readStream
         .format("cloudFiles")
         .option("cloudFiles.format", "json")
         .option(
             "cloudFiles.schemaLocation",
             schema_path
         )
         .load(source_path)
)

In [0]:
raw_stream.printSchema()

root
 |-- actor: string (nullable = true)
 |-- created_at: string (nullable = true)
 |-- id: string (nullable = true)
 |-- org: string (nullable = true)
 |-- payload: string (nullable = true)
 |-- public: string (nullable = true)
 |-- repo: string (nullable = true)
 |-- type: string (nullable = true)
 |-- _rescued_data: string (nullable = true)



Bronze Table Name

In [0]:
bronze_table = (
    f"{CATALOG}.{BRONZE_SCHEMA}.github_events_bronze"
)

print(bronze_table)

de_portfolio.bronze.github_events_bronze


Write Stream to Bronze

In [0]:
dbutils.fs.rm(
    checkpoint_path,
    recurse=True
)

True

In [0]:
display(
    dbutils.fs.ls(
        f"{RAW_VOLUME_PATH}/checkpoints"
    )
)

[]

In [0]:
query = (
    raw_stream.writeStream
        .format("delta")
        .option(
            "checkpointLocation",
            checkpoint_path
        )
        .trigger(availableNow=True)
        .toTable(bronze_table)
)

Wait for Completion

In [0]:
query.awaitTermination()

Verify Bronze Table

In [0]:
spark.sql(f"""
SELECT *
FROM {bronze_table}
LIMIT 10
""").show(truncate=False)

+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------------+-----------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------+-------------------------------------------------------------------------------------------------------------------------------------------------+-----------+-------------+
|actor                                                                                                                                      

In [0]:
display(
    dbutils.fs.ls(checkpoint_path)
)

path,name,size,modificationTime
dbfs:/Volumes/de_portfolio/raw/github_volume/checkpoints/bronze/commits/,commits/,0,1781129304074
dbfs:/Volumes/de_portfolio/raw/github_volume/checkpoints/bronze/metadata,metadata,45,1781129279000
dbfs:/Volumes/de_portfolio/raw/github_volume/checkpoints/bronze/offsets/,offsets/,0,1781129304074
dbfs:/Volumes/de_portfolio/raw/github_volume/checkpoints/bronze/sources/,sources/,0,1781129304075


In [0]:
test_df = (
    spark.read.format("json")
         .load(source_path)
)

print(test_df.count())

360


In [0]:
spark.sql(f"""
SELECT COUNT(*)
FROM {bronze_table}
""").show()

+--------+
|COUNT(*)|
+--------+
|     630|
+--------+



In [0]:
display(
    spark.table(
        "de_portfolio.bronze.github_events_bronze"
    )
)

actor,created_at,id,org,payload,public,repo,type,_rescued_data
"{""id"":33753566,""login"":""Andre-Durante"",""display_login"":""Andre-Durante"",""gravatar_id"":"""",""url"":""https://api.github.com/users/Andre-Durante"",""avatar_url"":""https://avatars.githubusercontent.com/u/33753566?""}",2026-06-10T21:49:38Z,13132804310,null,"{""ref"":""main"",""ref_type"":""branch"",""full_ref"":""refs/heads/main"",""master_branch"":""main"",""description"":""NxtTune lets gym members vote on what plays next, creating a workout soundtrack powered by the crowd."",""pusher_type"":""user""}",true,"{""id"":1265493250,""name"":""Andre-Durante/NxtTune"",""url"":""https://api.github.com/repos/Andre-Durante/NxtTune""}",CreateEvent,null
"{""id"":292306026,""login"":""jbarroso168"",""display_login"":""jbarroso168"",""gravatar_id"":"""",""url"":""https://api.github.com/users/jbarroso168"",""avatar_url"":""https://avatars.githubusercontent.com/u/292306026?""}",2026-06-10T21:49:38Z,13132804707,null,"{""repository_id"":1264614930,""push_id"":35421581130,""ref"":""refs/heads/main"",""head"":""9424d236c3410e4a034496f98f0dfcf0b2eab70d"",""before"":""b012cef7c83246b2848b23cb8bc18b46452eb32a""}",true,"{""id"":1264614930,""name"":""jbarroso168/gymtrack"",""url"":""https://api.github.com/repos/jbarroso168/gymtrack""}",PushEvent,null
"{""id"":41898282,""login"":""github-actions[bot]"",""display_login"":""github-actions"",""gravatar_id"":"""",""url"":""https://api.github.com/users/github-actions[bot]"",""avatar_url"":""https://avatars.githubusercontent.com/u/41898282?""}",2026-06-10T21:49:38Z,13132804691,null,"{""repository_id"":1213340588,""push_id"":35421581142,""ref"":""refs/heads/main"",""head"":""9a6437f5d92b254d1d70a3b0cd34394d55a8f90b"",""before"":""02e06ca2a7c02fd96d223412c7b5a69deef539cf""}",true,"{""id"":1213340588,""name"":""IgnacioR04/TFG_TERMINAL_FINAL"",""url"":""https://api.github.com/repos/IgnacioR04/TFG_TERMINAL_FINAL""}",PushEvent,null
"{""id"":39297556,""login"":""heyitsmeharv"",""display_login"":""heyitsmeharv"",""gravatar_id"":"""",""url"":""https://api.github.com/users/heyitsmeharv"",""avatar_url"":""https://avatars.githubusercontent.com/u/39297556?""}",2026-06-10T21:49:38Z,13132804690,null,"{""repository_id"":1255171032,""push_id"":35421580975,""ref"":""refs/heads/main"",""head"":""dbea4871052aa7cd5c51afa076b0200af811ea4b"",""before"":""ff00a7169e6a665cfbcb585fab776b4d2bda8c08""}",true,"{""id"":1255171032,""name"":""heyitsmeharv/my-tight-five"",""url"":""https://api.github.com/repos/heyitsmeharv/my-tight-five""}",PushEvent,null
"{""id"":10352239,""login"":""freggy"",""display_login"":""freggy"",""gravatar_id"":"""",""url"":""https://api.github.com/users/freggy"",""avatar_url"":""https://avatars.githubusercontent.com/u/10352239?""}",2026-06-10T21:49:38Z,13132804668,"{""id"":138144191,""login"":""spacechunks"",""gravatar_id"":"""",""url"":""https://api.github.com/orgs/spacechunks"",""avatar_url"":""https://avatars.githubusercontent.com/u/138144191?""}","{""ref"":""frg/adjust-run-flavor-version-endpoint"",""ref_type"":""branch"",""full_ref"":""refs/heads/frg/adjust-run-flavor-version-endpoint"",""master_branch"":""main"",""description"":""Host and explore Minecraft minigames"",""pusher_type"":""user""}",true,"{""id"":726227166,""name"":""spacechunks/explorer"",""url"":""https://api.github.com/repos/spacechunks/explorer""}",CreateEvent,null
"{""id"":160979109,""login"":""Bubba5136"",""display_login"":""Bubba5136"",""gravatar_id"":"""",""url"":""https://api.github.com/users/Bubba5136"",""avatar_url"":""https://avatars.githubusercontent.com/u/160979109?""}",2026-06-10T21:49:38Z,13132804667,null,"{""repository_id"":1190043627,""push_id"":35421580906,""ref"":""refs/heads/main"",""head"":""49c6a395513f70257fbdb33db9a2477a70825e03"",""before"":""2ed99e065fb8d4cdae1b343e816eaa71c09dba6b""}",true,"{""id"":1190043627,""name"":""Bubba5136/Heavyoverhaul"",""url"":""https://api.github.com/repos/Bubba5136/Heavyoverhaul""}",PushEvent,null
"{""id"":275262016,

In [0]:
spark.sql("""
SELECT COUNT(*)
FROM de_portfolio.bronze.github_events_bronze
""").show()

+--------+
|COUNT(*)|
+--------+
|    1890|
+--------+

